In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
)

RANDOM_STATE = 42
TARGET = "Churn"

project_dir = Path.cwd().resolve()
if not (project_dir / "data").is_dir() and (project_dir.parent / "data").is_dir():
    project_dir = project_dir.parent

DATA_DIR = project_dir / "data"
MODEL_DIR = project_dir / "models"
REPORT_DIR = project_dir / "reports"

for output_dir in (MODEL_DIR, REPORT_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)

print("Imports and project paths initialized.")

In [ ]:
data_path = DATA_DIR / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(data_path)

print(data_path, df.shape)

display(df.head())

print("\nDataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum())

print("\nChurn distribution:")
display(df[TARGET].value_counts())

print("\nChurn percentage:")
display(
    (df[TARGET].value_counts(normalize=True) * 100).round(2)
)

In [3]:
# Convert TotalCharges into numeric format.
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce",
)

# Remove customer identifier.
df = df.drop(
    columns=["customerID"],
    errors="ignore",
)

# Encode target.
df[TARGET] = df[TARGET].map({
    "No": 0,
    "Yes": 1,
})

if df[TARGET].isna().any():
    raise ValueError("Unexpected values found in Churn.")

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

print("Features:", X.shape)
print("Target:", y.shape)

print("\nMissing values after conversion:")
display(X.isnull().sum()[X.isnull().sum() > 0])

Features: (7043, 19)
Target: (7043,)

Missing values after conversion:


TotalCharges    11
dtype: int64

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining churn rate:", round(y_train.mean(), 3))
print("Testing churn rate:", round(y_test.mean(), 3))

Training features: (5634, 19)
Testing features: (1409, 19)

Training churn rate: 0.265
Testing churn rate: 0.265


In [5]:
def make_preprocessor(input_frame):

    numeric_columns = (
        input_frame
        .select_dtypes(include="number")
        .columns
        .tolist()
    )

    categorical_columns = (
        input_frame
        .select_dtypes(exclude="number")
        .columns
        .tolist()
    )

    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns,
            ),
        ]
    )

    return preprocessor


def make_model_pipeline(input_frame, estimator):

    return Pipeline(steps=[
        ("preprocess", make_preprocessor(input_frame)),
        ("model", estimator),
    ])


print("Preprocessing functions created.")

Preprocessing functions created.


In [6]:
lr_options = {
    "Unweighted": None,
    "Balanced": "balanced",
}

cv_strategy_day1 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

day1_trials = {}

for option_name, weight in lr_options.items():

    print("Testing:", option_name)

    candidate = make_model_pipeline(
        X_train,
        LogisticRegression(
            max_iter=2000,
            class_weight=weight,
            random_state=RANDOM_STATE,
        ),
    )

    # Out-of-fold probabilities from training data.
    oof_probabilities = cross_val_predict(
        candidate,
        X_train,
        y_train,
        cv=cv_strategy_day1,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    # Calculate precision and recall at different thresholds.
    precisions, recalls, thresholds = precision_recall_curve(
        y_train,
        oof_probabilities,
    )

    # Calculate F1 for each available threshold.
    f1_scores = (
        2 * precisions[:-1] * recalls[:-1]
        / (
            precisions[:-1]
            + recalls[:-1]
            + 1e-10
        )
    )

    best_index = np.argmax(f1_scores)

    day1_trials[option_name] = {
        "weight": weight,
        "threshold": thresholds[best_index],
        "precision": precisions[best_index],
        "recall": recalls[best_index],
        "f1": f1_scores[best_index],
    }

print("\nCross-validation results:")

display(
    pd.DataFrame(day1_trials).T
)

Testing: Unweighted
Testing: Balanced

Cross-validation results:


,weight,threshold,precision,recall,f1
Unweighted,NaN,0.330384,0.560568,0.739799,0.637832
Balanced,balanced,0.574173,0.557837,0.745151,0.63803


In [7]:
best_option = max(
    day1_trials,
    key=lambda name: day1_trials[name]["f1"],
)

best_config = day1_trials[best_option]

baseline_threshold = float(
    best_config["threshold"]
)

print("Selected configuration:", best_option)

print(
    "Selected threshold:",
    round(baseline_threshold, 4),
)

print(
    "Validation precision:",
    round(best_config["precision"], 4),
)

print(
    "Validation recall:",
    round(best_config["recall"], 4),
)

print(
    "Validation F1:",
    round(best_config["f1"], 4),
)

Selected configuration: Balanced
Selected threshold: 0.5742
Validation precision: 0.5578
Validation recall: 0.7452
Validation F1: 0.638


In [8]:
baseline_pipeline = make_model_pipeline(
    X_train,
    LogisticRegression(
        max_iter=2000,
        class_weight=best_config["weight"],
        random_state=RANDOM_STATE,
    ),
)

baseline_pipeline.fit(
    X_train,
    y_train,
)

print("Improved Day 1 Logistic Regression fitted.")

Improved Day 1 Logistic Regression fitted.


In [9]:
# Predict probabilities.
test_probabilities = baseline_pipeline.predict_proba(
    X_test
)[:, 1]

# Apply selected threshold.
test_predictions = (
    test_probabilities >= baseline_threshold
).astype(int)

# Calculate metrics.
day1_metrics = {
    "Model": "Day 1 Logistic Regression",
    "Accuracy": accuracy_score(
        y_test,
        test_predictions,
    ),
    "Precision": precision_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),
    "Recall": recall_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),
    "F1": f1_score(
        y_test,
        test_predictions,
        zero_division=0,
    ),
    "ROC_AUC": roc_auc_score(
        y_test,
        test_probabilities,
    ),
}

display(
    pd.DataFrame([day1_metrics]).set_index("Model")
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        test_predictions,
        target_names=["No Churn", "Churn"],
        zero_division=0,
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        test_predictions,
    )
)

,Accuracy,Precision,Recall,F1,ROC_AUC
Model,,,,,
Day 1 Logistic Regression,0.762243,0.538767,0.724599,0.618016,0.841298



Classification Report:
              precision    recall  f1-score   support

    No Churn       0.89      0.78      0.83      1035
       Churn       0.54      0.72      0.62       374

    accuracy                           0.76      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.79      0.76      0.77      1409


Confusion Matrix:
[[803 232]
 [103 271]]


In [ ]:
day1_model_path = (
    MODEL_DIR / "churn_day1_baseline.joblib"
)

day1_artifact = {
    "pipeline": baseline_pipeline,
    "threshold": baseline_threshold,
    "model_name": "Logistic Regression",
    "positive_class": 1,
}

joblib.dump(
    day1_artifact,
    day1_model_path,
)

print("Day 1 model saved:", day1_model_path)

In [ ]:
def predict_churn(customer_rows, artifact=None):

    if artifact is None:
        artifact = day1_artifact

    fitted_model = artifact["pipeline"]
    threshold = artifact["threshold"]

    probabilities = fitted_model.predict_proba(
        customer_rows
    )[:, 1]

    labels = (
        probabilities >= threshold
    ).astype(int)

    return pd.DataFrame(
        {
            "predicted_churn": labels,
            "churn_probability": probabilities,
        },
        index=customer_rows.index,
    )


display(
    predict_churn(X_test.head())
)

,predicted_churn,churn_probability
437,0,0.115472
2280,1,0.853079
2235,0,0.146831
4460,1,0.660663
3761,0,0.060436


In [ ]:
day1_report_path = (
    REPORT_DIR / "day1_baseline_metrics.csv"
)

pd.DataFrame([day1_metrics]).to_csv(
    day1_report_path,
    index=False,
)

print("Metrics saved:", day1_report_path)

# Day 2 — Compare Candidate Models

Goal: Compare Decision Tree, Random Forest, and SVM
against the Day 1 Logistic Regression baseline.

Selection metric: Churn Recall.

In [ ]:
day2_candidates = {
    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    ),

    "SVM": SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

print("Candidate models:", list(day2_candidates.keys()))

Candidate models: ['Decision Tree', 'Random Forest', 'SVM']


In [ ]:
cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

day2_fitted = {}
day2_rows = []

for model_name, estimator in day2_candidates.items():

    candidate_pipeline = make_model_pipeline(
        X_train,
        estimator
    )

    # Calculate cross-validation recall using training data
    cv_recall = cross_val_score(
        candidate_pipeline,
        X_train,
        y_train,
        scoring="recall",
        cv=cv_strategy,
        n_jobs=-1
    ).mean()

    # Train on the complete training dataset
    candidate_pipeline.fit(X_train, y_train)

    day2_fitted[model_name] = candidate_pipeline

    # Evaluate on the untouched test dataset
    predictions = candidate_pipeline.predict(X_test)
    probabilities = candidate_pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "Model": model_name,
        "CV_Recall": cv_recall,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, probabilities)
    }

    day2_rows.append(metrics)

day2_comparison = pd.DataFrame(day2_rows)

day2_comparison = day2_comparison.sort_values(
    by="CV_Recall",
    ascending=False
)

display(day2_comparison)

,Model,CV_Recall,Accuracy,Precision,Recall,F1,ROC_AUC
2,SVM,0.779253,0.743790,0.511504,0.772727,0.615548,0.821065
1,Random Forest,0.632116,0.767211,0.554245,0.628342,0.588972,0.822399
0,Decision Tree,0.490970,0.731725,0.494565,0.486631,0.490566,0.653696


In [ ]:
day2_comparison.to_csv(
    REPORT_DIR / "day2_model_comparison.csv",
    index=False
)

print("Day 2 comparison report saved successfully.")

Day 2 comparison report saved successfully.


In [ ]:
day2_champion_name = day2_comparison.iloc[0]["Model"]

day2_champion = day2_fitted[day2_champion_name]

print("Day 2 Champion:", day2_champion_name)

print("Selection basis: Highest cross-validation recall.")

Day 2 Champion: SVM
Selection basis: Highest cross-validation recall.


In [ ]:
champion_predictions = day2_champion.predict(X_test)

print("Classification Report:")
print(
    classification_report(
        y_test,
        champion_predictions,
        target_names=["No Churn", "Churn"]
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, champion_predictions))

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.90      0.73      0.81      1035
       Churn       0.51      0.77      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.75      0.71      1409
weighted avg       0.80      0.74      0.76      1409

Confusion Matrix:
[[759 276]
 [ 85 289]]


In [ ]:
day2_champion_path = MODEL_DIR / "churn_day2_champion.joblib"

joblib.dump(
    day2_champion,
    day2_champion_path
)

print("Champion saved at:", day2_champion_path)

Champion saved at: C:\Users\KIIT\Desktop\RetailMind\models\churn_day2_champion.joblib


# Day 3 — Hyperparameter tuning and generalisation


In [ ]:
day3_search_spaces = {
    "Decision Tree": (
        DecisionTreeClassifier(
            random_state=RANDOM_STATE,
            class_weight="balanced"
        ),
        {
            "model__max_depth": [4, 8, None],
            "model__min_samples_leaf": [1, 5],
        },
    ),

    "Random Forest": (
        RandomForestClassifier(
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
        ),
        {
            "model__n_estimators": [150, 250],
            "model__max_depth": [None, 10],
            "model__min_samples_leaf": [1, 3],
        },
    ),

    "SVM": (
        SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced"
        ),
        {
            "model__C": [0.5, 1, 3],
            "model__gamma": ["scale", "auto"],
        },
    ),
}

print("Search spaces defined.")